# CNOT Gate

An example of using the Logical Assembler to generate a circuit containing a logical CNOT.

In [ ]:
# This file contains information which is proprietary to Riverlane Ltd
# ("Riverlane") and is Riverlane Confidential Information.
# (c) Copyright Riverlane 2025-2026. All rights reserved.
from typing import Literal
from deltakit_compile.frontend.logasm import LogAsmBuilder, LogAsmProgram, RotatedPlanarPatch

In [2]:
def build_cnot(
    width: int,
    height: int,
    prep_bases: tuple[Literal["X", "Y", "Z"], Literal["X", "Y", "Z"]] = ("Z", "Z"),
    meas_bases: tuple[Literal["X", "Y", "Z"], Literal["X", "Y", "Z"]] = ("Z", "Z"),
) -> LogAsmProgram:
    """
    Build a LogASM program that implements a logical CNOT gate.

    Args:
        width: The width of the patches involved.
        height: The height of the patches involved.
        prep_bases: Preparation bases of the control and target patches respectively.
        meas_bases: Measurement bases of the control and target patches respectively.

    Returns:
        Instantiated subroutine object to be provided to the LogicalAssembler.
    """
    builder = LogAsmBuilder()
    lq0 = builder.declare_patch(
        RotatedPlanarPatch(width=width, height=height, location=(0, 0), vertical_z=False)
    )
    anc = builder.declare_patch(
        RotatedPlanarPatch(width=width, height=height, location=(width + 1, 0), vertical_z=False)
    )
    lq1 = builder.declare_patch(
        RotatedPlanarPatch(
            width=width, height=height, location=(width + 1, height + 1), vertical_z=False
        )
    )
    bridge0anc = builder.declare_patch(
        RotatedPlanarPatch(width=1, height=height, location=(width, 0), vertical_z=False)
    )
    bridge1anc = builder.declare_patch(
        RotatedPlanarPatch(width=width, height=1, location=(width + 1, height), vertical_z=False)
    )
    stab_rounds = max(width, height)
    # Patch prepares - these are arbitrary for lq0 and lq1, necessarily X for ancilla
    lq0.prepare(prep_bases[0])
    lq1.prepare(prep_bases[1])
    anc.prepare("X")

    # Measure stabilisers
    lq0.measure_stabilisers(stab_rounds)
    lq1.measure_stabilisers(stab_rounds)
    anc.measure_stabilisers(stab_rounds)

    # Measure lq1 an extra 5 rounds
    lq1.measure_stabilisers(stab_rounds)

    # Multi Pauli measurement
    builder.multi_pauli_measure(
        operand_patches=[lq0, anc],
        bridges=[bridge0anc],
        rounds=stab_rounds,
        pauli_bases=["Z", "Z"],
    )

    # Measure lq0 an extra 5 rounds
    lq0.measure_stabilisers(stab_rounds)

    # Multi Pauli measurement
    builder.multi_pauli_measure(
        operand_patches=[lq1, anc],
        bridges=[bridge1anc],
        rounds=stab_rounds,
        pauli_bases=["X", "X"],
    )

    # Measure ancilla qubit (fixed basis - ancilla)
    anc.measure("Z")

    # Measure lq0/lq1 for an extra d rounds
    lq1.measure_stabilisers(stab_rounds)
    lq0.measure_stabilisers(stab_rounds)

    # Measure lq0 and lq1 (arbitrary basis)

    lq1.measure(meas_bases[0])
    lq0.measure(meas_bases[1])

    return builder.build_program()

In [3]:
cnot_gate = build_cnot(3, 3, ("Z", "Z"), ("Z", "Z"))
print(cnot_gate)

LogAsmProgram({
  %qreg = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>
  %qreg_1 = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(3, 3), location=(4.0, 0.0), orient=h_z>
  %qreg_2 = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(3, 3), location=(4.0, 4.0), orient=h_z>
  %qreg_3 = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(1, 3), location=(3.0, 0.0), orient=h_z>
  %qreg_4 = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(3, 1), location=(4.0, 3.0), orient=h_z>
  %qreg_5 = log_asm.prepare<Z> (%qreg : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>)
  %qreg_6 = log_asm.prepare<Z> (%qreg_2 : !log_asm.patch.rot_planar<size=(3, 3), location=(4.0, 4.0), orient=h_z>)
  %qreg_7 = log_asm.prepare<X> (%qreg_1 : !log_asm.patch.rot_planar<size=(3, 3), location=(4.0, 0.0), orient=h_z>)
  %qreg_8 = log_asm.meas_stab<3> (%qreg_5 : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>)
 